## Embeddings

Los embeddings de texto son representaciones numéricas de un texto que capturan su significado de una manera que las máquinas pueden entender. Son una forma de convertir palabras, frases o incluso documentos completos en una lista de números (vectores) que codifican su significado semántico.  

Por ejemplo, las palabras "rey" y "reina" pueden tener embeddings similares porque ambas representan la realeza, mientras que "manzana" y "naranja" pueden tener embeddings que reflejen su similitud como frutas.  

En este ejemplo empleamos un modelo LLM preentrenado para generar el embedding.

## 🧩 ¿Qué son los embeddings y por qué son importantes?

### El problema de representar texto para máquinas

Las máquinas no entienden palabras, solo números. Las representaciones clásicas como **one-hot** o **TF-IDF** tienen limitaciones fundamentales:

| Representación | Ventaja | Problema |
|---|---|---|
| One-hot | Simple | No captura similitud semántica. "perro" y "gato" son ortogonales |
| TF-IDF | Ponderación por frecuencia | Alta dimensionalidad. No hay noción de significado |
| **Embedding** | **Captura semántica** | **Necesita entrenamiento (o modelo preentrenado)** |

### La intuición geométrica

Un embedding es un **mapa semántico**: palabras con significados similares quedan cerca en el espacio vectorial. Esto permite hacer "aritmética de palabras":

```
vector("rey") - vector("hombre") + vector("mujer") ≈ vector("reina")
```

> Esta propiedad emergente no se programa explícitamente — el modelo la aprende de grandes volúmenes de texto.

---
### 🏗️ Cómo se generan los embeddings

En este notebook usamos **`all-MiniLM-L6-v2`**, un modelo Sentence-BERT preentrenado que genera embeddings de **384 dimensiones** para frases completas (no solo palabras). Ha sido entrenado en millones de pares de frases para maximizar la similitud entre frases semánticamente relacionadas.

In [2]:
from sentence_transformers import SentenceTransformer
import tensorflow as tf

# Cargar el modelo preentrenado para generar embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')  # Modelo preentrenado disponible en hugging face. Es bastante ligero

# Lista de frases para convertir a embeddings
frases = [
    "La inteligencia artificial está transformando el mundo.",
    "El aprendizaje profundo es una técnica clave en el procesamiento de lenguaje natural.",
    "¿Cómo se generan los embeddings?",
    "en que consisten los embeddings y lod transformers",
    "La inteligencia artificial es una tecnica transformadora"
]

# Generar embeddings para las frases
embeddings = model.encode(frases)

# Mostrar los resultados
for i, frase in enumerate(frases):
    print(f"Frase: {frase}")
    print(f"Embedding: {embeddings[i][:5]}...")  # Mostrar solo los primeros valores del embedding
    print()


c:\Users\tomas\envs\dl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\tomas\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tomas\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mod

Frase: La inteligencia artificial está transformando el mundo.
Embedding: [-0.04948277  0.05770751 -0.00050066 -0.05088013 -0.00792877]...

Frase: El aprendizaje profundo es una técnica clave en el procesamiento de lenguaje natural.
Embedding: [ 0.01676626  0.02484795 -0.05824437 -0.01149721 -0.06505699]...

Frase: ¿Cómo se generan los embeddings?
Embedding: [ 0.0096938  -0.09212607  0.01370838  0.00419838  0.00237615]...

Frase: en que consisten los embeddings y lod transformers
Embedding: [-0.07904126 -0.04865227  0.00054113 -0.08442481  0.01466055]...

Frase: La inteligencia artificial es una tecnica transformadora
Embedding: [-0.08353412  0.01540009 -0.05721198 -0.07926245 -0.05903133]...



### 📦 Modelo preentrenado: `all-MiniLM-L6-v2`

Este modelo de Hugging Face es una versión comprimida de BERT optimizada para velocidad:
- **384 dimensiones** por embedding (frente a las 768 de BERT base)
- **6 capas transformer** (frente a las 12 de BERT base)
- **~80MB** de tamaño — muy ligero para producción
- Excelente para tareas de **similitud semántica** y **búsqueda de información**

In [3]:
embeddings[2].shape

(384,)

#### Calculo de similaridad entre frases

---
## 📐 Similitud del coseno

La forma más natural de comparar dos embeddings es el **coseno del ángulo** entre ellos:

$$\text{similitud}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

- **1.0**: idénticos (mismo significado)
- **0.0**: ortogonales (sin relación semántica)
- **-1.0**: opuestos semánticamente

**¿Por qué coseno y no distancia euclídea?**
Los embeddings tienen normas variables. El coseno normaliza por la magnitud, midiendo solo la *dirección* (el significado), no la *intensidad*.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
similitudes = cosine_similarity(embeddings)
# Mostrar los resultados
print("Matriz de similitud del coseno:")
print(similitudes)

# Interpretar similitudes para pares específicos
print("\nSimilitud entre la frase 1 y 2:", similitudes[0][1])
print("Similitud entre la frase 1 y 5:", similitudes[0][4])
print("Similitud entre la frase 2 y 5:", similitudes[1][4])
print("Similitud entre la frase 3 y 4:", similitudes[2][3])

Matriz de similitud del coseno:
[[1.0000001  0.47234815 0.38231775 0.39474615 0.80503994]
 [0.47234815 0.99999964 0.40248802 0.25172287 0.44661933]
 [0.38231775 0.40248802 1.0000002  0.6916104  0.31252074]
 [0.39474615 0.25172287 0.6916104  1.0000004  0.43628487]
 [0.80503994 0.44661933 0.31252074 0.43628487 1.0000005 ]]

Similitud entre la frase 1 y 2: 0.47234815
Similitud entre la frase 1 y 5: 0.80503994
Similitud entre la frase 2 y 5: 0.44661933
Similitud entre la frase 3 y 4: 0.6916104


### 🔍 Interpretando los resultados

Observa cómo el modelo captura relaciones semánticas que van más allá de las palabras exactas:
- Frases sobre "IA transformando el mundo" y "IA técnica transformadora" → **alta similitud** aunque usan palabras distintas
- "¿Cómo se generan los embeddings?" y "en qué consisten los embeddings y los transformers" → **similitud media** (tema relacionado, perspectiva distinta)

Este es el poder clave frente a TF-IDF: capturar **sinónimos, paráfrasis y contexto**.

---
## 🎬 Caso práctico: Reseñas de FilmAffinity

Aplicamos el mismo modelo a reseñas reales de películas españolas. Esto nos permite:
1. Calcular qué reseñas son semánticamente similares entre sí
2. Encontrar las reseñas más representativas de cada película
3. Agrupar reseñas por temática sin necesidad de etiquetas

> 💡 **Aplicación práctica**: esto es exactamente lo que hacen los motores de recomendación de contenido — representan items y usuarios en un mismo espacio vectorial para calcular afinidad.

#### Ejemplo con las reseñas de peliculas españolas

In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
import numpy as np

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tomas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
path='./data/reviews_filmaffinity.csv'
df2 = pd.read_table(path, sep='\|\|', header=0, engine='python')
df2.sample(5)

<>:2: SyntaxWarning: invalid escape sequence '\|'
<>:2: SyntaxWarning: invalid escape sequence '\|'
C:\Users\tomas\AppData\Local\Temp\ipykernel_71252\704397863.py:2: SyntaxWarning: invalid escape sequence '\|'
  df2 = pd.read_table(path, sep='\|\|', header=0, engine='python')


,film_name,gender,film_avg_rate,review_rate,review_title,review_text
4846,"Torrente, el brazo tonto de la ley",Comedia,"5,8",7.0,Y Segura se inmortalizó,"Santiago Segura perpetraba, como él mismo decí..."
3435,Alatriste,Aventuras,"5,5",3.0,Mucho ruido y ninguna nuez,"Cuando uno va a ver esta película, después de ..."
1806,Ágora,Aventuras,"6,5",9.0,"Gracias, Alejandro",Me gustaría simplemente dar las gracias al Ale...
7269,REC,Terror,"6,6",1.0,Vergüenza ajena.,¿Qué hace la pava de los cuarenta principales ...
7881,Las 13 rosas,Drama,"6,6",6.0,Docudrama aceptable,"Pues bien, por falta de dinero o interés, el c..."


In [7]:
df3=df2[df2['film_name']=='Tadeo Jones 2']
df3.count()

film_name        36
gender           36
film_avg_rate    36
review_rate      36
review_title     36
review_text      36
dtype: int64

In [8]:
df3['review_text'].tolist()

['"¿Qué es eso que me llama este señor de gitanilla?" Momia.Con cinco años desde la primera entrega, "Las aventuras de Tadeo Jones", Enrique Gato nos trae de nuevo al albañil arqueólogo en una de sus aventuras infantiles. Para analizar una película como "Tadeo Jones 2: El tesoro del Rey Midas" no puede limitarse uno a un análisis intrínseco: esta película es española, y como tal ha tenido más dificultades para existir que una película nacida en Pixar, DreamWorks o cualquier compañía similar.La historia nos devuelve a la mayoría de los personajes de la primera entrega, a excepción del vendedor ambulante a quien ponía voz José Mota. Un taxista andaluz ha sido la elección para sustituirlo.El resto todos siguen como estaban, no es una película que vaya a destacar especialmente por su desarrollo de personajes: Tadeo y Sara siguen como en la anterior con ese pequeña subtrama romántica a la que se nota que quieren dar más calado, la incorporación del taxista es acertada y hay un descubrimient

In [9]:
embeddingsTAdeo = model.encode(df3['review_text'].tolist())

In [10]:
df3['review_text'][3237]

'l simpático aventurero de producción española da un salto de calidad con una secuela que mejora sustancialmente la primera parte a nivel visual, con una animación más elaborada y detallada, y que también ofrece una historia mucho más divertida y con mejor ritmo en localizaciones tanto exóticas como cercanas.A todo esto hay que sumarle el cambio de secundario chistoso, que esta vez recae en el personaje de "La Momia", quien da más juego y tiene más gracia que su predecesor. En definitiva, el director Enrique Gato y su equipo demuestran que aún tienen mucho que contar sobre este aprendiz de arqueólogo y que son capaces de seguir mejorando en el modo de hacerlo. Más mini críticas en cinedepatio.com'

In [11]:
embeddingsTAdeo[1]

array([ 2.08656583e-02,  1.76261291e-02, -1.99322421e-02, -3.15782614e-02,
       -2.67101079e-02, -5.36230067e-03,  4.86004688e-02,  6.34678975e-02,
        1.86383184e-02,  6.33155927e-02,  1.27927527e-01, -8.27723816e-02,
        2.65720803e-02, -1.76671874e-02,  1.54092275e-02, -9.81142931e-03,
        5.64837195e-02,  2.71039493e-02,  5.65596111e-03,  2.63696499e-02,
        1.09973297e-01, -1.17674835e-01, -2.01914348e-02,  6.47431165e-02,
       -9.78938639e-02,  5.15736565e-02,  1.30422590e-02,  1.36492364e-02,
       -1.22715503e-01, -8.97514150e-02, -5.74698634e-02,  1.74350142e-02,
        7.63321817e-02, -2.29309425e-02, -2.57971343e-02,  5.02237417e-02,
       -4.36918903e-03, -3.96816358e-02, -3.61933671e-02,  7.41332620e-02,
       -9.12410989e-02,  4.87463027e-02, -7.59974718e-02, -4.61738110e-02,
       -1.66564304e-02, -8.77921358e-02,  7.26263449e-02,  2.07646191e-02,
       -3.95855643e-02, -2.94245444e-02, -9.01828855e-02, -5.48268259e-02,
       -6.11939989e-02, -